In [3]:
# import libaries
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# change the current working dir
os.chdir('/home/saad-naeem/Desktop/Level_4/Graduation_project_saad/chatbot/Baytology')

In [3]:
# read dataset as dataframe
df = pd.read_csv('../Baytology/DataSet/egypt_real_estate_listings.csv')

In [4]:
# explore data columns
print(f"df_columns: {df.columns}")

# insight -> we have 11 columns

df_columns: Index(['url', 'price', 'description', 'location', 'type', 'size', 'bedrooms',
       'bathrooms', 'available_from', 'payment_method', 'down_payment'],
      dtype='object')


In [5]:
# data describtion
df.info()

# insight -> we should convert every column to spesific datatype

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19924 entries, 0 to 19923
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   url             19924 non-null  object
 1   price           19385 non-null  object
 2   description     19846 non-null  object
 3   location        19833 non-null  object
 4   type            19847 non-null  object
 5   size            19847 non-null  object
 6   bedrooms        19780 non-null  object
 7   bathrooms       19784 non-null  object
 8   available_from  19261 non-null  object
 9   payment_method  19383 non-null  object
 10  down_payment    5445 non-null   object
dtypes: object(11)
memory usage: 1.7+ MB


In [6]:
# see the null values
df.isna().sum()

# insight -> we have to fill this null values

url                   0
price               539
description          78
location             91
type                 77
size                 77
bedrooms            144
bathrooms           140
available_from      663
payment_method      541
down_payment      14479
dtype: int64

In [7]:
f = df['down_payment'].isna().sum() 
print(f)

14479


In [8]:
# explore 'available_from' column
df['available_from']

0        31 Aug 2025
1         2 Sep 2025
2        19 Aug 2025
3        26 Aug 2025
4         2 Sep 2025
            ...     
19919    21 Aug 2025
19920     1 Sep 2025
19921    30 Jul 2025
19922    23 Aug 2025
19923    21 Aug 2025
Name: available_from, Length: 19924, dtype: object

# Data Exploration

# Data Cleaning

In [9]:
# Clean 'size' (Extract sqm numbers)
# This regex finds digits before " sqm"
df['size_sqm'] = df['size'].astype(str).str.extract(r'([\d,]+)\s*sqm')
df['size_sqm'] = pd.to_numeric(df['size_sqm'].str.replace(',', ''), errors='coerce')

# Fallback: If sqm is missing, try to find sqft and convert
mask_missing = df['size_sqm'].isna()
df.loc[mask_missing, 'temp_sqft'] = pd.to_numeric(
    df.loc[mask_missing, 'size'].astype(str).str.extract(r'([\d,]+)\s*sqft')[0].str.replace(',', ''), 
    errors='coerce'
)
df['size_sqm'] = df['size_sqm'].fillna(df['temp_sqft'] / 10.764)

In [10]:
# # Fallback: If sqm is missing, try to find sqft and convert
# mask_missing = df['size_sqm'].isna()
# df.loc[mask_missing, 'temp_sqft'] = pd.to_numeric(
#     df.loc[mask_missing, 'size'].astype(str).str.extract(r'([\d,]+)\s*sqft')[0].str.replace(',', ''), 
#     errors='coerce'
# )
# df['size_sqm'] = df['size_sqm'].fillna(df['temp_sqft'] / 10.764)

In [11]:
# Clean 'price' (remove commas, convert to number)
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(',', '', regex=False), errors='coerce')

In [12]:
# Clean 'down_payment'
df['down_payment'] = pd.to_numeric(df['down_payment'].astype(str).str.replace(',', '', regex=False), errors='coerce')

In [13]:
# Clean 'bedrooms' and 'bathrooms' (extract first number found)
df['bedrooms'] = pd.to_numeric(df['bedrooms'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
df['bathrooms'] = pd.to_numeric(df['bathrooms'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

In [14]:
# see the null values
df.isna().sum()

# insight -> we have to fill this null values

url                   0
price               539
description          78
location             91
type                 77
size                 77
bedrooms            472
bathrooms           145
available_from      663
payment_method      541
down_payment      19924
size_sqm             77
temp_sqft         19924
dtype: int64

# --- Feature Engineering & Encoding ---


In [15]:
# Simplify 'location': Keep top 10, mark rest as 'Other'
top_10_locs = df['location'].value_counts().nlargest(10).index
df['location'] = df['location'].where(df['location'].isin(top_10_locs), 'Other')

In [16]:
# Drop columns not useful for the model
df_clean = df.drop(columns=['url', 'description', 'available_from', 'size', 'temp_sqft','down_payment','temp_sqft'], errors='ignore')
df_clean.to_csv('egypt_real_estate_preprocessed.csv', index=False)

In [17]:
# One-Hot Encode Categorical Variables (Type, Location, Payment Method)
df_encoded = pd.get_dummies(df_clean, columns=['location', 'type', 'payment_method'], drop_first=True)

# --- Handle Missing Values ---

In [18]:
# Decision Trees in sklearn cannot handle NaNs, so we fill them.
numerical_cols = ['price', 'size_sqm', 'bedrooms', 'bathrooms', 'down_payment']
for col in numerical_cols:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].fillna(df_encoded[col].median())

In [19]:
# Final Check
print(f"Final shape: {df_encoded.shape}")
print(f"Remaining missing values: {df_encoded.isnull().sum().sum()}")

Final shape: (19924, 31)
Remaining missing values: 0


In [20]:
# Save
# df_encoded.to_csv('egypt_real_estate_preprocessed.csv', index=False)

# see data STD 

In [1]:
import pandas as pd

In [6]:
df_2 = pd.read_csv("../Baytology/notebooks/egypt_real_estate_preprocessed_analysis-and-segmentation.csv")

In [7]:
df_2.head()

,url,price,description,location,type,size,bedrooms,bathrooms,available_from,payment_method,...,unit_sqft,size_sqm,unit_sqm,mid_room,compound,district,city,governorate,price_per_sqm,price_per_sqft
0,https://www.propertyfinder.eg/en/plp/buy/chale...,8000000.0,OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION...,"Swan Lake Gouna, Al Gouna, Hurghada, Red Sea",Chalet,732 sqft / 68 sqm,1,1.0,31 Aug 2025,Cash,...,sqft,68.0,sqm,1,Swan Lake Gouna,Al Gouna,Hurghada,Red Sea,117647.058824,10928.961749
1,https://www.propertyfinder.eg/en/plp/buy/villa...,25000000.0,"For sale, a villa with immediate delivery in C...","Karmell, New Zayed City, Sheikh Zayed City, Giza",Villa,"2,368 sqft / 220 sqm",4,4.0,2 Sep 2025,Cash,...,sqft,220.0,sqm,0,Karmell,New Zayed City,Sheikh Zayed City,Giza,113636.363636,10557.432432
2,https://www.propertyfinder.eg/en/plp/buy/chale...,15135000.0,"With a down payment of EGP 1,513,000, a fully ...","Azha North, Ras Al Hekma, North Coast",Chalet,"1,270 sqft / 118 sqm",2,2.0,19 Aug 2025,Cash,...,sqft,118.0,sqm,0,Azha North,Ras Al Hekma,North Coast,"New Cairo City, Cairo",128262.711864,11917.322835
3,https://www.propertyfinder.eg/en/plp/buy/apart...,12652000.0,Own an apartment in New Cairo with a minimal d...,"Taj City, 5th Settlement Compounds, The 5th Se...",Apartment,"1,787 sqft / 166 sqm",3,2.0,26 Aug 2025,Installments,...,sqft,166.0,sqm,0,Taj City,5th Settlement Compounds,The 5th Settlement,"New Cairo City, Cairo",76216.867470,7080.022384
4,https://www.propertyfinder.eg/en/plp/buy/villa...,45250000.0,Project: Granville\nLocation: Fifth Settlement...,"Granville, New Capital City, Cairo",Villa,"4,306 sqft / 400 sqm",7,7.0,2 Sep 2025,Cash,...,sqft,400.0,sqm,0,Granville,New Capital City,Cairo,"New Cairo City, Cairo",113125.000000,10508.592661


In [19]:
unique_values = []
for x in df_2.columns:
    unique_values.append(f"{x}: {df_2[x].unique()}")
    unique_values.append(" ")


with open('columns_unique_values.txt', 'w') as file:
    # f"\n{x}: {df_2[x].unique()}"
    file.writelines(unique_values)

In [20]:
unique_values

["url: ['https://www.propertyfinder.eg/en/plp/buy/chalet-for-sale-red-sea-hurghada-al-gouna-swan-lake-gouna-7841194.html'\n 'https://www.propertyfinder.eg/en/plp/buy/villa-for-sale-giza-sheikh-zayed-city-new-zayed-city-karmell-7855294.html'\n 'https://www.propertyfinder.eg/en/plp/buy/chalet-for-sale-north-coast-ras-al-hekma-azha-north-7779188.html'\n ...\n 'https://www.propertyfinder.eg/en/plp/buy/chalet-for-sale-north-coast-markaz-al-hamam-white-sand-7837951.html'\n 'https://www.propertyfinder.eg/en/plp/buy/villa-for-sale-cairo-new-cairo-city-the-5th-settlement-mostakbal-city-compounds-sarai-7796571.html'\n 'https://www.propertyfinder.eg/en/plp/buy/chalet-for-sale-north-coast-qesm-ad-dabaah-mountain-view-7788956.html']",
 ' ',
 'price: [ 8000000. 25000000. 15135000. ...  5704000.  7340006.   350000.]',
 ' ',
 'description: ["OWN A CHALET IN EL GOUNA WITH A PRIME LOCATION, FULL SEA VIEW, AND LUXURIOUS FINISHES.\\nDEVOLOBER : ORASCOM\\nUNITTYPE l CHALET\\nDETAILS :\\n- 1 BEDROOM ( MASTE